# 04 — Performance without self-deception
**JupyterSharp** kernel (C#) on the **.NET 11** runtime. Prerequisite: `dotnet build src/MathemaSharp.slnx -c Release`. From a terminal: `scripts/run-notebooks.ps1` (sets the roll-forward to runtime 11); in VS Code you need `DOTNET_ROLL_FORWARD=LatestMajor` and, as long as .NET 11 is RC, `DOTNET_ROLL_FORWARD_TO_PRERELEASE=1`.

The notebook explores correctness and environment; it does NOT replace BenchmarkDotNet.

In [ ]:
#load "../shared/MathemaSharp.csx"
using System.Numerics;
using MathemaSharp.Core;
using System.Runtime.InteropServices;
Console.WriteLine(RuntimeInformation.FrameworkDescription);
Console.WriteLine($"SIMD={Vector.IsHardwareAccelerated}, double lanes={Vector<double>.Count}");
var data = Enumerable.Range(0,4099).Select(i => (i%17)*0.125).ToArray();
Console.WriteLine(NumericAlgorithms.Dot<double>(data,data));
Console.WriteLine(Signals.VectorDot(data,data));

.NET 11.0.0-rc.3.26522.148
SIMD=True, double lanes=4
5633.390625
5633.390625

## From a terminal, at the repository root
```sh
dotnet run -c Release --project src/benchmarks/MathemaSharp.Benchmarks -- --filter '*SumBenchmarks*' --job Short --artifacts ops/artifacts/benchmarks
dotnet run -c Release --project src/benchmarks/MathemaSharp.Benchmarks -- --filter '*DotBenchmarks*' --job Short --artifacts ops/artifacts/benchmarks
```
Benchmarks: manual/generic/LINQ/Kahan sum, float/double/decimal, scalar/SIMD dot, BigInteger factorial, Complex DFT. Inputs prepared in GlobalSetup, results returned, MemoryDiagnoser on.
Compare Mean, Error, Ratio and Allocated for the same N; note CPU, runtime, JIT and GC from the report. `--job Dry` is only a smoke test: a single cold-start sample does not allow conclusions. The notebook kernel is a Roslyn scripting process, not a measurement environment. Generic Math does not imply SIMD; SIMD reorders the reduction.